### Display MTEB results
MTEB results are saved in .json files, one for each task. 
This notebook aggregates the results for multiple tasks and multiple models.

In [ ]:
import os
os.getcwd()

In [ ]:
import mteb
import json
import pandas as pd
import numpy as np

In [ ]:
# get task type 
mteb.get_task("ArguAna").metadata.type

#### Analyze MTEB results of different TF-IDF configurations

In [ ]:
data = {}
main_dir = "../MTEB/sparse_results"

# Traverse folders and subfolders
for (root, dirs, files) in os.walk(main_dir):
    # Identify model version from the folder structure
    model_version = os.path.basename(root)
    print(model_version)
        
    for file in files:
        # Skip unwanted files
        if file in {"model_meta.json"} or not file.endswith('.json'):
            continue
            
        file_path = os.path.join(root, file)
        try:
            # Read JSON and extract necessary fields
            with open(file_path, 'r') as f:
                json_data = json.load(f)
                task_name = json_data.get("task_name")
                main_score = json_data.get("scores", {}).get("test", {})[0]["main_score"]
                main_score = round(main_score*100, 2)
                    
                if task_name and main_score is not None:
                    if task_name not in data:
                        data[task_name] = {}
                    data[task_name][model_version] = main_score
        except (json.JSONDecodeError, KeyError):
            print(f"Error parsing file: {file_path}")


In [ ]:
df = pd.DataFrame.from_dict(data, orient='index')
df.index.name = "task_name"
df["task_types"] = [mteb.get_task(task).metadata.type for task in df.index]
df

In [ ]:
task_selection = ["ArguAna", "ArxivClusteringP2P", "BiorxivClusteringP2P", "MedrxivClusteringP2P", "MindSmallReranking",
                 "RedditClusteringP2P", "SCIDOCS", "SciDocsRR", "StackExchangeClusteringP2P", "STS15", "STS16",
                 "STSBenchmark"]

df.loc[task_selection].sort_values("task_types")

In [ ]:
df[["svd50_log", "svd_log_run2", "svd200_log", "svd300_log", "svd500_log", "task_types"]].loc[task_selection].sort_values("task_types")

In [ ]:
df[["tfidf_log_run2", "svd_log_old_run1", "svd_log_run2", "svd_log_piecewise", "task_types"]].loc[task_selection].sort_values("task_types")

#### Analyze kNN results of different TF-IDF configurations

In [ ]:
data = {}
main_dir = "../MTEB/knn_results_custom_vocab"

# Traverse folders and subfolders
for (root, dirs, files) in os.walk(main_dir):
        
    for file in files:
        # Skip unwanted files
        model_name = file.strip(".json")
            
        file_path = os.path.join(root, file)
        try:
            # Read JSON and extract necessary fields
            with open(file_path, 'r') as f:
                json_data = json.load(f)
                for key, value in json_data.items():
                    if type(value) == list:
                        value = np.mean(value)
                    json_data[key] = round(value*100, 2)

                data[model_name] = json_data
        except (json.JSONDecodeError, KeyError):
            print(f"Error parsing file: {file_path}")


In [ ]:
pd.DataFrame.from_dict(data)